In [1]:
import os
import logging
import onnx
import torch
from pathlib import Path
import matplotlib.pyplot as pyplot
import numpy as np
import torchvision
from torchvision import tv_tensors
from torchvision.io import read_image, ImageReadMode
from torchvision.models import resnet50
from torchvision.models.detection.backbone_utils import _resnet_fpn_extractor, _validate_trainable_layers
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor, MaskRCNN
import torch.nn as nn
from torchvision.ops.boxes import masks_to_boxes
from torchvision.transforms import v2
from torchvision.utils import draw_segmentation_masks, draw_bounding_boxes
import datetime
from inference_training import initCudaEnvironment

In [2]:
initCudaEnvironment(numCudaDevices=1,
                    visibleCudaDevices="0",
                    clearCudaDeviceCount=False)


In [3]:

model = torchvision.models.detection.maskrcnn_resnet50_fpn()

fileName = "onnx_export_test"
torchFileName = fileName + ".pt"
onnxFileName = fileName + ".onnx"


In [4]:

torch.save(model.state_dict(), torchFileName)

In [5]:
device = torch.device('cpu')
onnx_input = torch.randn(1,3, 200, 200)
model.to(device)
model.eval()
predictions = model(onnx_input)

torch.onnx.export(model, onnx_input, onnxFileName, export_params=True,  opset_version=20, dynamo=False )


/tmp/ipykernel_29898/2563339383.py:7: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, onnx_input, onnxFileName, export_params=True,  opset_version=20, dynamo=False )
/home/frank/opt/miniforge3/envs/tygronai/lib/python3.11/site-packages/torch/nn/functional.py:5185: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  * torch.tensor(scale_factors[i], dtype=torch.float32)
/home/frank/opt/miniforge3/envs/tygronai/lib/python3.11/site-packages/torchvision/ops/boxes.py:180: UserWarning: To copy con

In [6]:

onnx_model = onnx.load(onnxFileName)